# Reproduce — Plain Q-Former · Amazon-Book (nhánh `feat/book`)

Tái lập pipeline **Amazon-Book** từ đầu trên `config_book.yaml`, nhánh `feat/book` của CoQLLM.
Kiến trúc **plain BLIP-2 Q-Former** (giống ML-1M): truy vấn chỉ cross-attention vào vector CF,
**không** instruction đi vào đường nuôi LLM.

**Đã áp cho book:** D2/D3 (`item_noun=book`, bỏ genres hằng số) · valid_small (eval nhanh,
CoLLM-parity — đổi checkpoint-selection nên giữ cố định giữa các run so sánh).

**Thứ tự:** R0 MF → R1 Stage-1 → R2 Stage-2 → R3 Stage-3 Step-1 (LoRA) →
R5 Stage-3 Step-2 (arm chính **B3-qformer-ucq**, `user_conditioned=True`).
Eval: 4.1 (step 2, kết quả chính) + 4.3 B0-textonly (step 1).

> **B2-qformer** (tắt `user_conditioned`) chưa tách thư mục trong `config_book`
> (`output_dir` cố định `..._book_vicuna`, không nội suy `${arm}`). Muốn chạy ablation B2
> phải thêm arm-interpolation vào `run.qformer_stage3_step2.output_dir` như bản ML-1M,
> nếu không B2 sẽ ĐÈ lên checkpoint B3.


# 0. Môi trường

## 0.1 Clone

Repo trên GitHub tên `CoQLLM`, nhưng **phải clone vào thư mục `/content/CoQLLM`** — toàn bộ
đường dẫn trong `configs/config_book.yaml` (data, ckpt, prompts) hardcode tiền tố `/content/CoQLLM/`.

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub Personal Access Token (PAT): ")

!git clone https://{token}@github.com/QuocBaoBuiNguyen/CoQLLM.git /content/CoQLLM
%cd /content/CoQLLM
!git checkout feat/book
!git log --oneline -5

In [ ]:
# Chạy lại ô này mỗi khi push code mới lên nhánh.
!git -C /content/CoQLLM pull origin feat/book

## 0.2 Nạp lại module + PYTHONPATH

In [ ]:
import sys
import importlib
from types import ModuleType

# Vá `imp` (bị gỡ khỏi Python 3.12) cho các thư viện cũ còn import nó.
imp = ModuleType('imp')
imp.reload = importlib.reload
sys.modules['imp'] = imp

In [ ]:
%env PYTHONPATH=src:$PYTHONPATH

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0.3 Miniconda

In [ ]:
!wget -c https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/Miniconda3.sh
!chmod +x /content/Miniconda3.sh
!bash /content/Miniconda3.sh -b -f -p /usr/local

In [ ]:
import sys
sys.path.append('/usr/local/lib/python3.12/site-packages')

!conda --version
!source /usr/local/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!source /usr/local/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

## 0.4 Khôi phục env `sigllm` từ cache trên Cloudflare R2

Không dựng bằng `conda env create -f env.yaml` — dựng từ đầu mất ~20 phút và `env.yaml`
đang pin `transformers==4.28.0` (bản đó **chưa có** `InstructBlipQFormerModel`, code sẽ không
import nổi). Cache tarball là môi trường thật đã dùng cho mọi kết quả.

In [ ]:
!pip install -q "docutils>=0.20,<0.22"
!pip install -q awscli

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=True)

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('R2_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('R2_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'auto'

CLOUDFLARE_ACCOUNT_ID = "b712ab0bb7699d3a04331f27f94ff98f"
os.environ['ENDPOINT_URL'] = f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com"

In [ ]:
%%bash
set -e
ENV_NAME="sigllm"

aws --endpoint-url="$ENDPOINT_URL" s3 cp "s3://sigllm/colab_conda_cache/${ENV_NAME}.tar.gz" "/content/${ENV_NAME}.tar.gz"

mkdir -p "/usr/local/envs/${ENV_NAME}"
tar -xzf "/content/${ENV_NAME}.tar.gz" -C "/usr/local/envs/${ENV_NAME}"
rm -f "/content/${ENV_NAME}.tar.gz"

source /usr/local/etc/profile.d/conda.sh
conda activate "$ENV_NAME"

## 0.5 Chốt phiên bản — **môi trường của hồ sơ**

`transformers==4.38.2` là bản đã sinh ra mọi số trong luận. Đừng nâng lên:
từ 4.45 HuggingFace sửa `Blip2QFormerModel` và đổi chữ ký `query_length`, hành vi sẽ lệch.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
python -m pip install -U "transformers==4.38.2" "accelerate==0.27.2"

In [ ]:
# Ghi lại phiên bản để dán vào bảng "môi trường thực nghiệm" của luận.
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
pip list | grep -Ei "^(transformers|peft|accelerate|torch|tokenizers|bitsandbytes|scikit-learn) "

# 1. Chuẩn bị dữ liệu (Amazon-Book)

In [ ]:
import sys
sys.path.insert(0, "src")

# Xoá sạch để chạy lại từ đầu. data_preprocessing_book.py sẽ giải nén các pkl
# (train/valid/test/valid_small_ood2) từ data/raw/amazon_book.zip.
!rm -rf /content/CoQLLM/data/processed/amazon-book/


## 1.1 Tiền xử lý chính (Amazon-Book)

Khác ML-1M (dựng từ `ratings.dat`): các split OOD của Amazon-Book đến từ **pickle dựng sẵn**
trong `data/raw/amazon_book.zip` (bám đúng split CoLLM/BinLLM). Script chỉ giải nén và chèn cột
`genres` tổng hợp để builder chạy được (cột này là hằng số → bị D3 bỏ khi render).


In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/datasets/data_preprocessing_book.py


## 1.2 Gắn nhãn warm / strict-cold cho tập test (Amazon-Book)


In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/datasets/preprocess_test_cold_warm_book.py


## 1.3 Dựng pickle cho Q-Former (book)

Dùng `config_book.yaml` → template Q-Former render với **`item_noun=book`** (fix D2) và **bỏ
genres hằng số** (D3). **Bắt buộc chạy trước R1** để Stage-1 học đúng chữ "book".


In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/build_qformer_dataset.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml


# 2. Trọng số LLM — Vicuna-7B-v1.5

Chép từ Drive (nhanh). Nếu chưa có trên Drive thì dùng ô `pull_llm_model.py` bên dưới để tải
từ HuggingFace rồi tự sao lưu lại.

In [ ]:
!mkdir -p /content/CoQLLM/ckpt/llm/base/
!cp -r /content/drive/MyDrive/CoQLLM/llm/vicuna-7b/* /content/CoQLLM/ckpt/llm/base/
!ls /content/CoQLLM/ckpt/llm/base/

In [ ]:
# !source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
# python /content/CoQLLM/src/coqllm/pipelines/llm/pull_llm_model.py

# 3. Huấn luyện

Thứ tự bắt buộc. **R0–R3 dùng chung cho mọi arm**; chỉ R4/R5 là rẽ nhánh khảo sát.

| | Bước | Huấn luyện cái gì | Arm |
|---|---|---|---|
| R0 | MF | embedding user/item cộng tác, sau đó **đông cứng** | chung |
| R1 | Stage-1 | Q-Former: ITC + ITM + ITG + item-item + user-item | chung |
| R2 | Stage-2 | Q-Former + `llm_proj`, mục tiêu sinh văn bản | chung |
| R3 | Stage-3 Step-1 | **chỉ LoRA**, prompt thuần chữ (Q-Former đóng băng) | → `B0-textonly` |
| R5 | Stage-3 Step-2 | Q-Former + `llm_proj`, prompt đầy đủ, LoRA đóng băng | `B3-qformer-ucq` |
| R4 | Stage-3 Step-2 | như trên, tắt điều kiện người dùng | `B2-qformer` |

## R0 — MF (nền cộng tác)

Sinh `e_u`, `e_i`. Mọi bước sau chỉ **đọc** MF, không cập nhật nó.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/rec/train_rec_baseline.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml

## R1 — Stage-1: căn chỉnh biểu diễn

Dạy Q-Former biến vector CF thành 8 soft token gắn được với ngữ nghĩa văn bản.
Năm mục tiêu: ITC (đối lập item↔text), ITM (khớp nhị phân, âm khó lấy từ ma trận ITC),
ITG (sinh văn bản có điều kiện), cộng item-item và user-item.

Đây là bước **chỉ Q-Former làm được** — một MLP không có nhánh text nên ITM/ITG không định
nghĩa được. Điểm này đáng nêu trong luận.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage1_representation.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml

## R2 — Stage-2: tiền huấn luyện sinh

Nối Q-Former vào LLM đông cứng và huấn luyện `llm_proj` bằng mục tiêu ngôn ngữ trên văn bản
item. Stage-2 gọi `encode_cf` (**không** có text) — trùng khớp với đường Stage-3 sau khi đã gỡ
instruction, nên `llm_proj` được huấn luyện và sử dụng trên cùng một phân phối đầu vào.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage2_generative.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml \
  --options run.qformer_stage2.lr=2e-4 \
            run.qformer_stage2.weight_decay=1e-4 \
            run.qformer_stage2.early_stopping_patience=8

## R3 — Stage-3 Step-1: LoRA trên prompt thuần chữ

Theo đúng quy trình hai bước của CoLLM. Prompt **không có** `<ItemIDList>` / `<TargetItemID>`,
nên LoRA học riêng phần tác vụ Yes/No mà chưa thấy soft token nào.

Checkpoint của bước này **chính là baseline `B0-textonly`** — đánh giá nó ở mục 4.3.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml \
  --options run.best_metric=uauc

## R5 — Stage-3 Step-2, arm `B3-qformer-ucq` (kết quả chính)

Nạp LoRA tốt nhất của R3 và **đóng băng nó**, rồi chỉ tinh chỉnh Q-Former + `llm_proj` trên
prompt đầy đủ. `user_conditioned=True` (mặc định trong config) nên 8 truy vấn được dịch thêm
`user_proj(user_cf)`: cùng một bộ phim sẽ được đọc bằng góc nhìn riêng của từng người dùng.

Ghi vào `ckpt/qformer_stage3_step2_book_vicuna/`.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config_book.yaml \
  --options run.best_metric=uauc

# 4. Đánh giá

Cả ba mục dưới đều chấm trên `test / test_warm / test_cold` và in AUC + uAUC.

## 4.1 `B3-qformer-ucq` — kết quả chính

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config_book.yaml \
  --step 2

## 4.3 `B0-textonly` — sàn không có tín hiệu cộng tác

Chấm thẳng checkpoint LoRA của R3 với `--step 1`: prompt thuần chữ, không soft token nào.
Đây là mức nền để trả lời "cầu nối CF đóng góp bao nhiêu".

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config_book.yaml \
  --step 1

# 5. Sao lưu checkpoint

Colab ngắt phiên là mất sạch. Chạy sau **mỗi** bước R0–R5, đừng để dồn.

In [ ]:
!mkdir -p /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!cp -r /content/CoQLLM/ckpt/mf_book                               /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!cp -r /content/CoQLLM/ckpt/qformer_stage1_book                   /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!cp -r /content/CoQLLM/ckpt/qformer_stage2_book_vicuna            /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!cp -r /content/CoQLLM/ckpt/qformer_stage3_step1_lora_book_vicuna /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!cp -r /content/CoQLLM/ckpt/qformer_stage3_step2_book_vicuna      /content/drive/MyDrive/CoQLLM/plain-qformer-book/
!ls -la /content/drive/MyDrive/CoQLLM/plain-qformer-book/


In [ ]:
# Bản sao lên Cloudflare R2 (bucket MỚI `coqllm`; conda_cache vẫn ở bucket cũ `sigllm`).
!aws --endpoint-url="$ENDPOINT_URL" s3 sync /content/CoQLLM/ckpt/ s3://coqllm/ckpt/